# Product Demand Forecasting - PySpark ML Pipeline

In [ ]:
# Check if the code is running in Colab
if 'google.colab' in str(get_ipython()):
  print('Running on CoLab')
  RunningInColab = True
else:
  print('Not running on CoLab')
  RunningInColab = False

Running on CoLab


In [ ]:
# Configure the following to use YOUR GCP setup

# 1. Configure the Project ID (not Project Name!!!) as per your GCP Dataproc setup
project_id = 'msandhu8-cis415-sum26a'

# 2. Configure Bucket name as per your Google Cloud Storage setup
bucket = 'msandhu8_data_for_gcp_labs'

# 3. Configure the path to the movie reviews data file as per your Google Cloud Storage setup
#    If your setup is exactly as per the instructions in GCP Lab 1c and in this lab:
#       --- you will not need to make any changes to the below line.
#    If your setup is different (due to whatever reason - doesn't matter),
#       --- just update the below line to reflect the path as per YOUR Google Cloud Storage setup
path_to_data_files = "/data_for_project/"

# 4. Configure the appropriate data file to be used for the task
#       Uncomment one of the two lines below based on the following:
#          In Google Colab, you should build/test with SMALL DATA
#          In GCP, first you should run with SMALL DATA
#             and finally, you should run with BIG DATA

# data_file_name = "ratings_big.txt"
data_file_name = "semiconductor_ai_demand_small.csv"

# Lastly, we will build the full path of the data file and confirm it's correct
# You do not need to change this line
relative_path_to_file = path_to_data_files[1:] + data_file_name #### removing the initial / character from path_to_data_files
full_file_path = "gs://" + bucket + "/" + relative_path_to_file

# Let's print out all the configurations and ensure that they are correct
print(f"ProjectID (and not the Project Name) is: {project_id}")
print(f"Bucket name is: {bucket}")

print(f"Relative path to the data file is {relative_path_to_file}")
print(f"Full path to the data file is {full_file_path}")

# You should not need to make any other change in the code below, unless your setup is different from the lab instructions

ProjectID (and not the Project Name) is: msandhu8-cis415-sum26a
Bucket name is: msandhu8_data_for_gcp_labs
Relative path to the data file is data_for_project/semiconductor_ai_demand_small.csv
Full path to the data file is gs://msandhu8_data_for_gcp_labs/data_for_project/semiconductor_ai_demand_small.csv


In [ ]:
# Typically, any big data platform (like GCP Dataproc) will have PySpark pre-installed
# PySpark may be pre-installed in Colab but not work perfectly.
# In all other platforms (e.g. your laptop), PySpark will be not pre-installed.
# This paragraph is to check if PySpark is available in the system and install if it's not available
# This paragraph will install PySpark if running in Colab as well.
# You should expect this paragraph NOT TO RUN the PySpark installation in GCP Dataproc

try:
  from pyspark.sql import SparkSession
  pyspark_available = 'Y'
except:
  pyspark_available = 'N'

if RunningInColab == True or pyspark_available == "N":
  # install pyspark
  print("Installing PySpark...")
  !apt-get update -qq
  !apt-get install -y openjdk-11-jdk-headless -qq
  # install pyspark
  !pip install -q pyspark
  print("PySpark installed")
else:
  # Spark / PySpark already pre-installed in the environment
  print("PySpark already pre-installed!")


Installing PySpark...
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Selecting previously unselected package openjdk-11-jre-headless:amd64.
(Reading database ... 118243 files and directories currently installed.)
Preparing to unpack .../openjdk-11-jre-headless_11.0.31+11-1ubuntu1~22.04.2_amd64.deb ...
Unpacking openjdk-11-jre-headless:amd64 (11.0.31+11-1ubuntu1~22.04.2) ...
Selecting previously unselected package openjdk-11-jdk-headless:amd64.
Preparing to unpack .../openjdk-11-jdk-headless_11.0.31+11-1ubuntu1~22.04.2_amd64.deb ...
Unpacking openjdk-11-jdk-headless:amd64 (11.0.31+11-1ubuntu1~22.04.2) ...
Setting up openjdk-11-jre-headless:amd64 (11.0.31+11-1ubuntu1~22.04.2) ...
update-alternatives: using /usr/lib/jvm/java-11-openjdk-amd64/bin/jjs to provide /usr/bin/jjs (jjs) in auto mode
update-alternatives: using /usr/lib/jvm/java-11-openjdk-am

In [ ]:
# Authorize access from Colab to GCP Cloud Storage
if RunningInColab == True:
  # To access Google Cloud Storage
  from google.cloud import storage
  import google.auth

  !pip install gcsfs
  import gcsfs

  from google.colab import auth
  auth.authenticate_user()

  credentials, default_project_id = google.auth.default()
  !gcloud config set project {project_id}

[environment: untagged] Read more to tag: g.co/cloud/project-env-tag.
Updated property [core/project].


To take a quick anonymous survey, run:
  $ gcloud survey



In [ ]:
# import the following from pyspark.sql
from pyspark.sql import SparkSession
from pyspark.sql.functions import lit
from pyspark.sql.functions import col
from pyspark.sql.types import *

print(f"Packages for PySpark imported")

if RunningInColab == True:
  # Create a spark session if running in colab.  In GCP, spark session is created automatically
  spark = SparkSession.builder.master("local[*]").getOrCreate()

  print("Spark Session created in Colab")

  # We need the appropriate package to access files in the google cloud storage from colab
  from google.cloud import storage

Packages for PySpark imported
Spark Session created in Colab


In [ ]:
project_id = "msandhu8-CIS415-SUM26A"

bucket = "msandhu8_data_for_gcp_labs"

path_to_data_files = "/data_for_project/"

data_file_name = "semiconductor_ai_demand_small.csv"

relative_path_to_file = path_to_data_files[1:] + data_file_name

full_file_path = "gs://" + bucket + "/" + relative_path_to_file

print(full_file_path)

gs://msandhu8_data_for_gcp_labs/data_for_project/semiconductor_ai_demand_small.csv


In [ ]:
import sys

RunningInColab = "google.colab" in sys.modules

print("Running in Colab:", RunningInColab)

Running in Colab: True


In [ ]:
print(f"Reading the data file: {full_file_path}")

if RunningInColab == True:

    # Copy the file from Google Storage to the local Colab runtime
    storage_client = storage.Client()

    temp_file_path = "/content/" + data_file_name

    bucket_object = storage_client.bucket(bucket)

    blob = bucket_object.blob(relative_path_to_file)

    blob.download_to_filename(temp_file_path)

    print("Reading local file using Spark")

    spark_df = spark.read.csv(
        temp_file_path,
        header=True,
        inferSchema=True
    )

else:

    print("Reading file directly from Google Cloud Storage")

    spark_df = spark.read.csv(
        full_file_path,
        header=True,
        inferSchema=True
    )

print("Spark dataframe loaded successfully.")

spark_df.show(5)

print(f"Total number of records = {spark_df.count()}")

# Use the same dataframe name as the rest of your notebook
df = spark_df

Reading the data file: gs://msandhu8_data_for_gcp_labs/data_for_project/semiconductor_ai_demand_small.csv
Reading local file using Spark
Spark dataframe loaded successfully.
+--------------+-----------------+-------+---------------+----------------+-------------------+--------------------+---------------------+-----------------------+--------------------+-------------+-------------+------------------+
|Product_Demand|Historical_Demand|  Price|Marketing_Spend|Seasonal_Factors|Economic_Conditions|Supply_Chain_Factors|Competitive_Landscape|Product_Lifecycle_Stage|Customer_Preferences|       Region|Sales_Channel|Account_Manager_ID|
+--------------+-----------------+-------+---------------+----------------+-------------------+--------------------+---------------------+-----------------------+--------------------+-------------+-------------+------------------+
|       46135.0|          51721.0|1060.97|      219826.22|               1|             109.76|               89.69|                6

In [ ]:
df.printSchema()


root
 |-- Product_Demand: double (nullable = true)
 |-- Historical_Demand: double (nullable = true)
 |-- Price: double (nullable = true)
 |-- Marketing_Spend: double (nullable = true)
 |-- Seasonal_Factors: integer (nullable = true)
 |-- Economic_Conditions: double (nullable = true)
 |-- Supply_Chain_Factors: double (nullable = true)
 |-- Competitive_Landscape: double (nullable = true)
 |-- Product_Lifecycle_Stage: integer (nullable = true)
 |-- Customer_Preferences: double (nullable = true)
 |-- Region: string (nullable = true)
 |-- Sales_Channel: string (nullable = true)
 |-- Account_Manager_ID: integer (nullable = true)



In [ ]:
df.describe().show()

+-------+------------------+------------------+-----------------+-----------------+------------------+-------------------+--------------------+---------------------+-----------------------+--------------------+-------------+-------------+------------------+
|summary|    Product_Demand| Historical_Demand|            Price|  Marketing_Spend|  Seasonal_Factors|Economic_Conditions|Supply_Chain_Factors|Competitive_Landscape|Product_Lifecycle_Stage|Customer_Preferences|       Region|Sales_Channel|Account_Manager_ID|
+-------+------------------+------------------+-----------------+-----------------+------------------+-------------------+--------------------+---------------------+-----------------------+--------------------+-------------+-------------+------------------+
|  count|             50000|             50000|            49991|            49991|             50000|              49994|               50000|                50000|                  50000|               50000|        50000|  

In [ ]:
print(df.count())

50000


In [ ]:
from pyspark.sql.functions import col, when, count

df.select([
    count(when(col(c).isNull(), c)).alias(c)
    for c in df.columns
]).show()

+--------------+-----------------+-----+---------------+----------------+-------------------+--------------------+---------------------+-----------------------+--------------------+------+-------------+------------------+
|Product_Demand|Historical_Demand|Price|Marketing_Spend|Seasonal_Factors|Economic_Conditions|Supply_Chain_Factors|Competitive_Landscape|Product_Lifecycle_Stage|Customer_Preferences|Region|Sales_Channel|Account_Manager_ID|
+--------------+-----------------+-----+---------------+----------------+-------------------+--------------------+---------------------+-----------------------+--------------------+------+-------------+------------------+
|             0|                0|    9|              9|               0|                  6|                   0|                    0|                      0|                   0|     0|            0|                 0|
+--------------+-----------------+-----+---------------+----------------+-------------------+-------------------

In [ ]:
# EDA
# EDA
from pyspark.sql.functions import *

df.groupBy("Seasonal_Factors").avg("Product_Demand").show()

+----------------+-------------------+
|Seasonal_Factors|avg(Product_Demand)|
+----------------+-------------------+
|               1|  47018.23272381857|
|               3|  52188.29033017018|
|               4|  54612.81998542392|
|               2|  49472.41990706774|
+----------------+-------------------+



In [ ]:
# Data Cleaning
from pyspark.sql.functions import mean

for c in ["Price", "Marketing_Spend", "Economic_Conditions"]:
    m = df.select(mean(c)).first()[0]
    df = df.fillna({c: m})

df.show(5)

+--------------+-----------------+-------+---------------+----------------+-------------------+--------------------+---------------------+-----------------------+--------------------+-------------+-------------+------------------+
|Product_Demand|Historical_Demand|  Price|Marketing_Spend|Seasonal_Factors|Economic_Conditions|Supply_Chain_Factors|Competitive_Landscape|Product_Lifecycle_Stage|Customer_Preferences|       Region|Sales_Channel|Account_Manager_ID|
+--------------+-----------------+-------+---------------+----------------+-------------------+--------------------+---------------------+-----------------------+--------------------+-------------+-------------+------------------+
|       46135.0|          51721.0|1060.97|      219826.22|               1|             109.76|               89.69|                63.29|                      2|               73.56|       Europe|  Distributor|              1111|
|       48386.0|          42380.0| 879.37|      245886.88|               2| 

In [ ]:
features = [
    "Historical_Demand",
    "Price",
    "Marketing_Spend",
    "Seasonal_Factors",
    "Economic_Conditions",
    "Supply_Chain_Factors",
    "Competitive_Landscape",
    "Product_Lifecycle_Stage",
    "Customer_Preferences"
]

In [ ]:
from pyspark.ml.feature import VectorAssembler

assembler = VectorAssembler(
    inputCols=features,
    outputCol="features"
)

data = assembler.transform(df).select("features", "Product_Demand")

data.show(5)

+--------------------+--------------+
|            features|Product_Demand|
+--------------------+--------------+
|[51721.0,1060.97,...|       46135.0|
|[42380.0,879.37,2...|       48386.0|
|[49160.0,836.04,2...|       57365.0|
|[46005.0,889.19,1...|       52695.0|
|[44102.0,824.31,2...|       48010.0|
+--------------------+--------------+
only showing top 5 rows


In [ ]:
train_df, val_df = data.randomSplit([0.8, 0.2], seed=42)

print("Training records:", train_df.count())
print("Validation records:", val_df.count())

Training records: 39951
Validation records: 10049


In [ ]:
from pyspark.ml.regression import LinearRegression

lr = LinearRegression(
    featuresCol='features',
    labelCol='Product_Demand'
)

lr_model = lr.fit(train_df)

lr_pred = lr_model.transform(val_df)

lr_pred.select("Product_Demand", "prediction").show(5)

+--------------+------------------+
|Product_Demand|        prediction|
+--------------+------------------+
|       22068.0| 23884.24500622873|
|       16244.0|25160.182298837502|
|       29306.0|28031.733602851906|
|       33290.0|32990.157280345644|
|       30066.0|27264.502448646614|
+--------------+------------------+
only showing top 5 rows


In [ ]:
from pyspark.ml.regression import RandomForestRegressor

rf = RandomForestRegressor(
    featuresCol='features',
    labelCol='Product_Demand',
    numTrees=50,
    seed=42
)

rf_model = rf.fit(train_df)

rf_pred = rf_model.transform(val_df)

rf_pred.select("Product_Demand", "prediction").show(5)

+--------------+-----------------+
|Product_Demand|       prediction|
+--------------+-----------------+
|       22068.0|44000.72564444322|
|       16244.0|43901.75485504369|
|       29306.0|45318.06057326282|
|       33290.0|46911.25539010428|
|       30066.0|45371.22474242121|
+--------------+-----------------+
only showing top 5 rows


In [ ]:
from pyspark.ml.evaluation import RegressionEvaluator

rmse = RegressionEvaluator(
    labelCol='Product_Demand',
    predictionCol='prediction',
    metricName='rmse'
)

mae = RegressionEvaluator(
    labelCol='Product_Demand',
    predictionCol='prediction',
    metricName='mae'
)

print("===== Linear Regression =====")
print("RMSE:", rmse.evaluate(lr_pred))
print("MAE :", mae.evaluate(lr_pred))

print()

print("===== Random Forest =====")
print("RMSE:", rmse.evaluate(rf_pred))
print("MAE :", mae.evaluate(rf_pred))

===== Linear Regression =====
RMSE: 5016.740415513166
MAE : 3985.2404599960323

===== Random Forest =====
RMSE: 6133.4386961146265
MAE : 4853.224358216597


After validating on the small dataset, deploy the pipeline to the big-data cluster. Run first with the small dataset in the cluster environment, then scale to the full 1M+ row dataset.